## energy_data file splitting 10_split_files.ipynb 
reading in energy_data.csv and preprocessing

### Summary
* Data in 15-minute intervals
* 64 organisations
* org_id, obj_id, period_begin, period_interval, value_type, value, value_quality

### Notebook content
* Goal: read in energy_data.csv and split into multiple files
* 01 look at the dataset with scan
* 02 read the dataset with read
* 03 split the dataset into 64 CSVs by org_id

### Findings
* Dataset has 541,445,297 rows and 7 columns
* too large to load directly, so polars is used
* for further processing, the data was split by org_id
*

#### Open

*

In [ ]:
#Imports
import polars as pl
import csv
from collections import defaultdict
import os

In [ ]:
#Konfig
data_file = 'data/energy_data.csv'
output_file = "data/data_ID_01.csv"
export = 0 # 1 = 1 file from scan, 2 = all files from scan, 3 = 1 file from df, 4 = all files from df

# 01 Scan

In [ ]:
# load file
# lazy read (does not load anything fully into memory)
df = pl.scan_csv(datei)

In [ ]:
df2 = df.head(5).collect()  # actually load 5 rows
print(df2)

In [ ]:
# Spaltennamen anzeigen
print(df.columns)

In [ ]:
# Schema (columns + data types)
print(df.schema)

In [ ]:
# row count (estimated or computed)
#rows = df.select(pl.len()).collect().item()
#print(f"Row count: {rows:,}")

# 02 Read

In [ ]:
df = pl.read_csv(datei)

In [ ]:
df.head(-1)

# 03 Split into csv's per org_id

### via scan

In [ ]:
# first id without loading df

if export == 1:
    # open LazyFrame (does not load everything into memory)
    df_teilen = pl.scan_csv(datei)

    # filter only rows with org_id == 1 and write directly
    (
        df
        .filter(pl.col("org_id") == 1)
        .sink_csv(output_file)
    )

In [ ]:
# all IDs

if export == 2:
    output_files = {}

    with open(datei, newline='', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)

        for row in reader:
            org_id = row["org_id"]
            if org_id not in output_files:
                out = open(f"data/split_files/org_{org_id}.csv", "w", newline='', encoding='utf-8')
                writer = csv.DictWriter(out, fieldnames=reader.fieldnames)
                writer.writeheader()
                output_files[org_id] = (out, writer)

            output_files[org_id][1].writerow(row)

    # close all output files
    for out, _ in output_files.values():
        out.close()

### via the read-in df

In [ ]:
# erste Id

if export == 3:
    df_filtered = df.filter(pl.col("org_id") == 1)
    df_filtered.write_csv(output_file)

In [ ]:
if export == 4:
    # ensure the target folder exists
    #os.makedirs("data/split_files", exist_ok=True)

    # get all unique org_ids
    org_ids = df["org_id"].unique().to_list()

    # write a separate CSV for each org_id
    for org_id in org_ids:
        print(f"Writing org_{org_id}.csv ...")
        df.filter(pl.col("org_id") == org_id).write_csv(f"data/split_files/org_{org_id}.csv")
        #print(org_id)



# Analysen

In [ ]:

org_ids = df["org_id"].unique().to_list()
summary_list = []

for org_id in org_ids:
    df_subset = df.filter(pl.col("org_id") == org_id)
    
    record_count = df_subset["value"].drop_nulls().len()   # <- changed here
    value_min = df_subset["value"].min()
    value_max = df_subset["value"].max()
    missing_value_count = df_subset["value"].null_count()
    
    summary_list.append(
        {
            "org_id": org_id,
            "record_count": record_count,
            "value_min": value_min,
            "value_max": value_max,
            "missing_value_count": missing_value_count
        }
    )

summary_df = pl.DataFrame(summary_list)

# save summary as csv
summary_df.write_csv("data/summary.csv")


print(summary_df)